In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [14]:
df = pd.read_csv('archive/dataset1.csv')
df.columns = df.columns.str.strip()
df = df.drop('index', axis=1)
y = df['Result']
X = df.drop('Result', axis=1)

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [16]:
gb_grid = GridSearchCV(GradientBoostingClassifier(random_state=42), {'n_estimators': [100, 200], 'max_depth': [3, 5], 'learning_rate': [0.1]}, cv=3, n_jobs=-1)
gb_grid.fit(X_train_s, y_train)
bp = gb_grid.best_params_
print(bp)

{'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}


In [17]:
#gradient boosting with feature engineering
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_train_fe = poly.fit_transform(X_train_s)
X_test_fe = poly.transform(X_test_s)
gb_fe = GradientBoostingClassifier(random_state=42, **bp).fit(X_train_fe, y_train)
pred_fe = gb_fe.predict(X_test_fe)
print(accuracy_score(y_test, pred_fe))
print(confusion_matrix(y_test, pred_fe, labels=[-1, 1]))
print(classification_report(y_test, pred_fe, digits=4))

0.9773857982813207
[[ 945   35]
 [  15 1216]]
              precision    recall  f1-score   support

          -1     0.9844    0.9643    0.9742       980
           1     0.9720    0.9878    0.9799      1231

    accuracy                         0.9774      2211
   macro avg     0.9782    0.9761    0.9770      2211
weighted avg     0.9775    0.9774    0.9774      2211



In [18]:
#gradient boosting with principle component analysis
pca = PCA(n_components=30, random_state=42)
X_train_pca = pca.fit_transform(X_train_s)
X_test_pca = pca.transform(X_test_s)
gb_pca = GradientBoostingClassifier(random_state=42, **bp).fit(X_train_pca, y_train)
pred_pca = gb_pca.predict(X_test_pca)
print(accuracy_score(y_test, pred_pca))
print(confusion_matrix(y_test, pred_pca, labels=[-1, 1]))
print(classification_report(y_test, pred_pca, digits=4))

0.9687924016282226
[[ 932   48]
 [  21 1210]]
              precision    recall  f1-score   support

          -1     0.9780    0.9510    0.9643       980
           1     0.9618    0.9829    0.9723      1231

    accuracy                         0.9688      2211
   macro avg     0.9699    0.9670    0.9683      2211
weighted avg     0.9690    0.9688    0.9687      2211

